In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
from typing import Dict, Tuple, Optional, Union, Any, List
import warnings
import scanpy as sc
warnings.filterwarnings('ignore')

# Import additional libraries for neural network training
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.metrics import precision_recall_curve, roc_curve, roc_auc_score, average_precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Input, Add, Activation, 
    MultiHeadAttention, LayerNormalization, Reshape, Flatten,
    GlobalAveragePooling1D, Embedding
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras import backend as K
from sklearn.ensemble import RandomForestClassifier

In [2]:
import scanpy.external as sce
import scanpy as sc

In [3]:
Apexigen = sc.read_h5ad('./Apexigen_data/Apexigen_gene_TCR_annotated_for_integration_real_case.h5ad')
Apexigen

AnnData object with n_obs × n_vars = 16286 × 15827
    obs: 'batch', 'n_counts', 'n_genes', 'outcome', 'patient', 'response', 'sample', 'timepoint', 'log_counts', 'mt_frac', 'scvi_r1', 'cell_class', 'cell_type', 'cell_type_scvi', 'cell_class_scvi', 'TRB_cdr3', 'TRA_cdr3'
    obsm: 'X_umap'
    layers: 'counts'

In [4]:
Apexigen.obs.to_csv('./Apexigen_data/Apexigen_obs_metadata_for_integration_real_case.csv')

In [13]:

gex = pd.read_csv('./Apexigen_data_pca_harmony_batch_correction_by_sample_embeddings.csv', index_col=0)
gex.shape

(16286, 50)

# PE + AE TCR

In [14]:
atchley = pd.read_csv("../data/atchley.txt", sep="\t")
# atchley['avg'] = atchley[['f1','f2','f3','f4','f5']] .sum(axis=0)
atchley
index_aa_converter = pd.DataFrame({"aa":atchley['amino.acid'].values, "pos": list(range(20))})

index_aa_converter.index = index_aa_converter['aa'].values
index_aa_converter.pop("aa")
index_aa_converter
atchley.pop('amino.acid')
atchley

cols = atchley.columns
cols
for col in cols:
    atchley[col] =  pd.to_numeric(atchley[col].str.replace("−", "-"), errors='coerce')
atchley['avg'] = atchley.sum(axis=1)
atchley
atchley_aa = pd.concat([index_aa_converter.reset_index(), atchley], axis=1)
atchley_aa
atchley_amino_prop = pd.concat([pd.DataFrame([[0]*atchley_aa.shape[1]],columns=atchley_aa.columns),atchley_aa], axis=0)
atchley_amino_prop
atchley_aa = atchley_amino_prop
atchley_aa.iloc[0,0]='*'
atchley_aa
# word_vectors = np.array(atchley_amino_prop)
word_vectors = np.array(atchley_amino_prop.iloc[:,2:8])
word_vectors.shape

(21, 6)

In [15]:
# We use aa that has length of at most 50
def get_atchley(receptor_seq, length = 35):
    res_atchley = np.zeros((length), dtype='int')
    # print(len(receptor_seq))
    if (len(receptor_seq) <= length):
        for l in range(len(receptor_seq)):
            # print(receptor_seq[l])
            if receptor_seq[l] in index_aa_converter.index:
                res_atchley[l] = 1+index_aa_converter.loc[receptor_seq[l]].values[0]
            else:
                res_atchley[l] = np.random.randint(0,20)
            # res_atchley[l,:] =  atchley.loc[index_aa_converter.loc[receptor_seq[l]].values[0]].values #atchley.loc[receptor_seq[l]]
            # res_atchley = res_atchley.append(atchley.loc[s])
    else:
        for l in range(length):
            # print(receptor_seq[l])
            if receptor_seq[l] in index_aa_converter.index:
                # print(index_aa_converter.loc[receptor_seq[l]].values[0])
                res_atchley[l] = 1+index_aa_converter.loc[receptor_seq[l]].values[0]
            else:
                res_atchley[l] = np.random.randint(0,20)
            # res_atchley[l,:] =  atchley.loc[index_aa_converter.loc[receptor_seq[l]].values[0]].values #atchley.loc[receptor_seq[l]]
            # res_atchley = res_atchley.append(atchley.loc[s])
    return res_atchley

In [16]:
# The input of AAEmbedding is the vectorization of amino acid sequence, i.e., the order of each amino acid letter base in the 
# list of 20 amino acid, we can obtain the input by using get_atchley function
class AAEmbedding(tf.keras.layers.Layer):
    def __init__(self, word_vectors):
        super(AAEmbedding, self).__init__()
        self.word_vectors = tf.constant(word_vectors, dtype=tf.float32)

    def call(self, inputs):
        embedded_inputs = tf.nn.embedding_lookup(self.word_vectors, inputs)
        return embedded_inputs


In [17]:
def positional_encoding(depth, length=25):
  depth = depth
  # print(depth)
  positions = np.arange(length)[:, np.newaxis]   
  positions = np.repeat(positions, depth, axis=1)
  # print(np.repeat(positions, 6, axis=1))
  # print(positions)  # (seq, 1)
  # depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)
  # print(depths.shape)
  angle_rates = 1 / (1000)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)
  # print("Angle radian", angle_rads.shape)
  s = np.sin(angle_rads)[::2]
  c = 1- np.cos(angle_rads)[1::2]
  # print(s)
  # print(c)

  # pos_encoding = np.concatenate(
  #     [np.sin(angle_rads), np.cos(angle_rads)],
  #     axis=-1) 
  pos_encoding = np.vstack(
      [s, c]) 
  return tf.cast(pos_encoding, dtype=tf.float32)



In [18]:
class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, d_model, word_vectors,length = 25):
    super().__init__()
    self.d_model = d_model
    self.length = length
    # self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True) 
    self.embedding = AAEmbedding(word_vectors)
    self.pos_encoding = positional_encoding(depth=d_model, length=length)

  def compute_mask(self, *args, **kwargs):
    return self.embedding.compute_mask(*args, **kwargs)

  def call(self, x):
    # length = tf.shape(x)[1]

    # print("PositionalEmbedding input shape", x)
    x = self.embedding(x)
    # print(f"beginning x shape {x.shape}")
    # This factor sets the relative scale of the embedding and positonal_encoding.
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    # print('self position encoding value',self.pos_encoding[tf.newaxis, :50, :].shape)
    # print("x value before last line PositionEmbedding",x.shape)
    # scale = tf.cast(x != 0, tf.float32)
    # print(self.pos_encoding[tf.newaxis, :25, :])
    # print(scale.shape)
    # po = self.pos_encoding[tf.newaxis, :25, :]
    # scale = tf.reshape(scale,po.shape)
    # print(po.shape)
    # x = x + tf.multiply(self.pos_encoding[tf.newaxis, :25, :],scale)
    x = x + self.pos_encoding[tf.newaxis, :self.length, :]
    # print("Finish Positional Embedding")
    return x


In [19]:
d_model = 6
pt = PositionalEmbedding(d_model=d_model,word_vectors=word_vectors,length=35)

In [20]:
AA_embed = AAEmbedding(word_vectors)
AA_eem = AA_embed(get_atchley("CASSLGTDTQYF"))
# pt(get_atchley("CASSLGTDTQYF"))

# We do for random split

In [21]:
Apexigen.obs

,batch,n_counts,n_genes,outcome,patient,response,sample,timepoint,log_counts,mt_frac,scvi_r1,cell_class,cell_type,cell_type_scvi,cell_class_scvi,TRB_cdr3,TRA_cdr3
AAACCTGAGGCGCTCT-1-4,4,1756.0,627,PD,05,pathCR,05_post,post_CD40,7.470794,0.032460,3,CD4T_naive,CD4T_naive3,CD4T_naive2,CD4T_naive,CASSVATAGGIGYTF,CAMREGWNTGTASKLTF
AAACCTGCATGCCACG-1-4,4,1787.0,707,PD,05,pathCR,05_post,post_CD40,7.488294,0.029659,2,CD4T_naive,CD4T_naive1,CD4T_naive1,CD4T_naive,CASSLGGVNTGELFF,CAASTGNDMRF
AAACCTGTCTCCTATA-1-6,6,1825.0,864,HD,HD_01,HD,HD_03,nan,7.509336,0.055342,3,CD4T_naive,CD4T_naive2,CD4T_naive2,CD4T_naive,CASSQGGLGLGGAVQPQHF,CAAGSGATNKLIF
AAACGGGAGGGATACC-1-8,8,974.0,470,PD,01,nopathCR,01_LTF2,follow_up,6.881412,0.075975,0,CD8T,CD8T_GrH1,CD8_GrH,CD8_T,CASSRLAEVNEQFF,CAESRSGGSYIPTF
AAACGGGCATCGGAAG-1-3,3,2800.0,865,PD,04,pathCR,04_surgery,surgery,7.937375,0.024286,7,CD4T_naive,CD4T_naive3,CD4T_naive3,CD4T_naive,CASSAGGARVSYEQYF,CAVFSGSRLTF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTCCTCCACTCGACG-1_05_post,0,4706.0,1357,PD,05,pathCR,05_post,post_CD40,8.456594,0.000000,1,CD4_Tcell,CD4T_naive,CD4_Tnaive1,CD4_T,CASSLDSYGYTF,CAVNMDRGSTLGRLYF
TTTGCGCGTGTGCCTG-1_05_post,0,2891.0,1380,PD,05,pathCR,05_post,post_CD40,7.969358,0.000000,3,CD8_Tcell,CD8T_GrK,CD8T_GrH,CD8_T,CASSLSYSEETQYF,CLVGERGSTLGRLYF
TTTGGTTCACCAGCAC-1_05_post,0,3474.0,1338,PD,05,pathCR,05_post,post_CD40,8.153062,0.000000,1,CD4_Tcell,CD4T_naive,CD4_Tnaive1,CD4_T,CASTQNGGDTQYF,CAVTLGGSYIPTF
TTTGGTTGTTCTGGTA-1_05_post,0,4632.0,1231,PD,05,pathCR,05_post,post_CD40,8.440744,0.000000,1,CD4_Tcell,CD4T_naive,CD4_Tnaive1,CD4_T,CASSNVRRSRDSPLHF,CAIRDGGAQKLVF


In [22]:
train_TCR_df = Apexigen.obs
train_TCR_df

,batch,n_counts,n_genes,outcome,patient,response,sample,timepoint,log_counts,mt_frac,scvi_r1,cell_class,cell_type,cell_type_scvi,cell_class_scvi,TRB_cdr3,TRA_cdr3
AAACCTGAGGCGCTCT-1-4,4,1756.0,627,PD,05,pathCR,05_post,post_CD40,7.470794,0.032460,3,CD4T_naive,CD4T_naive3,CD4T_naive2,CD4T_naive,CASSVATAGGIGYTF,CAMREGWNTGTASKLTF
AAACCTGCATGCCACG-1-4,4,1787.0,707,PD,05,pathCR,05_post,post_CD40,7.488294,0.029659,2,CD4T_naive,CD4T_naive1,CD4T_naive1,CD4T_naive,CASSLGGVNTGELFF,CAASTGNDMRF
AAACCTGTCTCCTATA-1-6,6,1825.0,864,HD,HD_01,HD,HD_03,nan,7.509336,0.055342,3,CD4T_naive,CD4T_naive2,CD4T_naive2,CD4T_naive,CASSQGGLGLGGAVQPQHF,CAAGSGATNKLIF
AAACGGGAGGGATACC-1-8,8,974.0,470,PD,01,nopathCR,01_LTF2,follow_up,6.881412,0.075975,0,CD8T,CD8T_GrH1,CD8_GrH,CD8_T,CASSRLAEVNEQFF,CAESRSGGSYIPTF
AAACGGGCATCGGAAG-1-3,3,2800.0,865,PD,04,pathCR,04_surgery,surgery,7.937375,0.024286,7,CD4T_naive,CD4T_naive3,CD4T_naive3,CD4T_naive,CASSAGGARVSYEQYF,CAVFSGSRLTF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTCCTCCACTCGACG-1_05_post,0,4706.0,1357,PD,05,pathCR,05_post,post_CD40,8.456594,0.000000,1,CD4_Tcell,CD4T_naive,CD4_Tnaive1,CD4_T,CASSLDSYGYTF,CAVNMDRGSTLGRLYF
TTTGCGCGTGTGCCTG-1_05_post,0,2891.0,1380,PD,05,pathCR,05_post,post_CD40,7.969358,0.000000,3,CD8_Tcell,CD8T_GrK,CD8T_GrH,CD8_T,CASSLSYSEETQYF,CLVGERGSTLGRLYF
TTTGGTTCACCAGCAC-1_05_post,0,3474.0,1338,PD,05,pathCR,05_post,post_CD40,8.153062,0.000000,1,CD4_Tcell,CD4T_naive,CD4_Tnaive1,CD4_T,CASTQNGGDTQYF,CAVTLGGSYIPTF
TTTGGTTGTTCTGGTA-1_05_post,0,4632.0,1231,PD,05,pathCR,05_post,post_CD40,8.440744,0.000000,1,CD4_Tcell,CD4T_naive,CD4_Tnaive1,CD4_T,CASSNVRRSRDSPLHF,CAIRDGGAQKLVF


In [23]:
train_TCR = pd.DataFrame()
for s in train_TCR_df.TRB_cdr3.values:
    train_TCR = pd.concat([train_TCR, pd.DataFrame(pt(get_atchley(s)).numpy().reshape(1,35*d_model))], axis=0)

In [24]:
train_TCR.to_csv(f"train_TCR_PE_only_no_AE_Apexigen.csv", index=False)

In [25]:

latent_dim = 128
input_dim = train_TCR.shape[1]
# Encoder
inp = keras.Input(shape=(input_dim,), name='encoder_input')
x = keras.layers.Dense(164)(inp)
x = keras.layers.BatchNormalization()(x)
latent = keras.layers.Dense(latent_dim, name="latent", activation="relu")(x)

# Decoder
x = keras.layers.Dense(164)(latent)
x = keras.layers.BatchNormalization()(x)
out = keras.layers.Dense(input_dim, activation="linear", name='decoder_output')(x) # Linear activation for reconstruction

# Model
model_random_split = keras.Model(inputs=inp, outputs=out, name="tcr_autoencoder")
model_random_split.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

In [26]:
model_random_split.fit(train_TCR, train_TCR, epochs=200, batch_size=64, validation_data=(train_TCR, train_TCR))

Epoch 1/200


2026-01-18 22:17:19.659632: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


255/255 [==============================] - 1s 2ms/step - loss: 3.9358 - val_loss: 0.6364
Epoch 2/200
255/255 [==============================] - 0s 1ms/step - loss: 0.5348 - val_loss: 0.2325
Epoch 3/200
255/255 [==============================] - 0s 1ms/step - loss: 0.3237 - val_loss: 0.1337
Epoch 4/200
255/255 [==============================] - 0s 1ms/step - loss: 0.2628 - val_loss: 0.0917
Epoch 5/200
255/255 [==============================] - 0s 1ms/step - loss: 0.2337 - val_loss: 0.0794
Epoch 6/200
255/255 [==============================] - 1s 2ms/step - loss: 0.2336 - val_loss: 0.0888
Epoch 7/200
255/255 [==============================] - 0s 2ms/step - loss: 0.2217 - val_loss: 0.0556
Epoch 8/200
255/255 [==============================] - 1s 2ms/step - loss: 0.2178 - val_loss: 0.0553
Epoch 9/200
255/255 [==============================] - 0s 2ms/step - loss: 0.2113 - val_loss: 0.0550
Epoch 10/200
255/255 [==============================] - 0s 2ms/step - loss: 0.2055 - val_loss: 0.0540
E

In [27]:
model_random_split.save_weights(f'./smart_aligned_v2_TCR_AE_PE_only_no_AE_Apexigen_weights.h5')

In [28]:
latent_model = keras.Model(model_random_split.input, model_random_split.get_layer("latent").output)
embeddings_TCR_PE = latent_model.predict(train_TCR, batch_size=64)

255/255 [==============================] - 0s 268us/step


In [29]:
np.save(f"train_TCR_with_PE_and_AE_embeddings_Apexigen.npy", embeddings_TCR_PE)

# Next,we would like to un ED for batch gene expression and AE of TCR (integration by ED)

# ED for raw gene random split

In [8]:
gex

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,1.087143,0.467443,2.992532,-0.600697,-3.994596,-3.333098,1.494854,-1.146100,1.571670,-3.118363,...,-0.210213,1.269835,0.131614,1.555858,-0.562283,0.154132,1.219355,-0.089831,-0.300811,0.098589
1,9.246613,-4.383909,-4.804466,-0.070527,1.494315,0.910335,2.323131,-1.229502,-1.463696,1.459822,...,0.890217,0.616105,-0.418397,0.040113,-0.075964,0.261941,1.278015,0.778815,-0.413370,0.533625
2,-4.378371,1.180819,-4.223704,-5.470704,0.444927,-0.000247,1.059516,0.668836,0.961870,1.295324,...,1.722274,0.045716,0.730128,1.226174,1.399147,-0.674245,0.482494,0.890334,-1.046426,-0.332977
3,-0.850734,5.800984,2.319241,0.621932,0.391961,1.471914,-1.820410,-1.887836,0.645741,-0.318717,...,-0.187072,0.493142,-0.507019,-0.700821,-0.482881,1.702497,-2.204038,0.750383,2.090266,-0.414377
4,7.510671,-3.492533,0.615445,-0.826791,-0.356948,-0.733857,-0.747724,-1.670011,-0.794903,-0.049055,...,0.420621,0.245370,-0.644681,-0.188387,-0.635079,0.767439,1.157366,0.416946,0.140676,-0.130281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,-0.928211,4.903222,-2.795438,-2.711269,-0.354280,-0.522040,-0.471457,-2.239837,-0.942964,0.537984,...,1.706733,-0.141238,1.290803,0.571046,2.032877,-0.091293,-2.120174,2.326794,-0.270818,2.348335
145475,-3.711599,5.466145,-1.662588,3.397898,-1.352260,2.653984,1.317506,2.093558,-0.908551,-1.140238,...,-0.622839,-0.013557,-0.371825,0.784568,-0.067285,-0.820746,-0.442909,-0.638775,-0.235357,-0.845757
145476,-0.427704,-4.652079,0.408053,-1.942324,-2.452670,2.487576,-1.482361,-0.260748,-0.498160,0.732487,...,0.760570,0.456670,-0.208458,-0.142902,-0.938427,0.216794,1.042893,-0.465932,-0.601186,0.570299
145477,7.317558,-4.430610,1.243477,0.069994,0.100587,-1.188651,-0.900679,-0.003285,-0.333207,-0.494998,...,1.442170,0.210964,0.092243,0.610511,-0.685681,-0.969873,-0.389187,0.062457,-0.254924,-0.485812


In [9]:

embeddings_TCR_PE = np.load(f'train_TCR_with_PE_and_AE_embeddings_Apexigen.npy')


In [14]:
train_input_raw = gex

In [11]:
from tensorflow.keras import layers, Model

hidden_dim = 64
lr=1e-3
input_dim = train_input_raw.shape[1]
output_dim = embeddings_TCR_PE.shape[1]
inp = layers.Input(shape=(input_dim,), name="raw_gex_input")

# Encoder: Raw counts → latent space
x = layers.Dense(input_dim, activation="relu")(inp)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(55, activation="tanh")(x)
x = layers.BatchNormalization()(x)
latent = layers.Dense(64, name="latent_layer")(x)

# Decoder: Latent → positional encoded TCR
d = layers.Dense(75, activation="relu")(latent)
d = layers.BatchNormalization()(d)
d = layers.Dropout(0.2)(d)
d = layers.Dense(96, activation="tanh")(d)
d = layers.BatchNormalization()(d)
out = layers.Dense(output_dim,name="pe_tcr_pred")(d)

model_ED_gex_batch_tcr = Model(inp, out, name="raw_gene2pe_tcr")
model_ED_gex_batch_tcr.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="mae", metrics=["mae"])

In [15]:
model_ED_gex_batch_tcr.fit(
    train_input_raw, embeddings_TCR_PE,
    validation_data=(train_input_raw, embeddings_TCR_PE),
    epochs=500,
    batch_size=128
)
# model_ED_gex_tcr.save_weights('./output/experiments/neg_ratio_3_1/merged_embeddings/smart_aligned_v2_model_ED_gex_tcr_ED_gex_to_tcr_model_random_split_1_weights_train.h5')
model_ED_gex_batch_tcr.save_weights(f'./smart_aligned_v2_model_ED_batch_gex_tcr_ED_gex_to_tcr_model_Apexigen.weights.h5')
latent_model_gex_tcr_integration = tf.keras.Model(model_ED_gex_batch_tcr.input, model_ED_gex_batch_tcr.get_layer("latent_layer").output)
ED_integration_gex_tcr = latent_model_gex_tcr_integration.predict(train_input_raw, batch_size=64)
np.save(f"ED_integration_batch_gex_tcr_embeddings_train_Apexigen.npy", ED_integration_gex_tcr)

Epoch 1/500


2026-01-19 11:18:21.387694: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


128/128 [==============================] - 1s 2ms/step - loss: 10.0800 - mae: 10.0800 - val_loss: 7.6361 - val_mae: 7.6361
Epoch 2/500
128/128 [==============================] - 0s 2ms/step - loss: 4.3112 - mae: 4.3112 - val_loss: 2.4140 - val_mae: 2.4140
Epoch 3/500
128/128 [==============================] - 0s 2ms/step - loss: 2.3258 - mae: 2.3258 - val_loss: 2.2827 - val_mae: 2.2827
Epoch 4/500
128/128 [==============================] - 0s 2ms/step - loss: 2.2838 - mae: 2.2838 - val_loss: 2.2608 - val_mae: 2.2608
Epoch 5/500
128/128 [==============================] - 0s 2ms/step - loss: 2.2681 - mae: 2.2681 - val_loss: 2.2463 - val_mae: 2.2463
Epoch 6/500
128/128 [==============================] - 0s 2ms/step - loss: 2.2507 - mae: 2.2507 - val_loss: 2.2307 - val_mae: 2.2307
Epoch 7/500
128/128 [==============================] - 0s 2ms/step - loss: 2.2345 - mae: 2.2345 - val_loss: 2.2143 - val_mae: 2.2143
Epoch 8/500
128/128 [==============================] - 0s 2ms/step - loss: 2.21